In [ ]:
# generated by axis/splitcausal_t4probe2.py -- do not edit
import hashlib
_PAYLOAD = '\nimport subprocess, sys\nif "tinygrad" not in sys.modules:\n    try:\n        import tinygrad  # noqa: F401\n    except ImportError:\n        print("tinygrad is not in this image; installing it", flush=True)\n        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tinygrad"])\n\nimport hashlib, json, os, re, statistics, time\nimport numpy as np\n\nDEV = os.environ.get("DEV", "CUDA")\nBUDGET_S = float(os.environ.get("PROBE_BUDGET_S", "1000"))\nT0 = time.time()\n\ndef log(*a): print(*a, flush=True)\ndef left(): return BUDGET_S - (time.time() - T0)\n\nimport tinygrad\nfrom tinygrad import Device, Tensor\nimport tinygrad.mixin.op as M\nimport tinygrad.tensor as TWMOD\n\nOP_PY = __import__("pathlib").Path(M.__file__)\nSRC = OP_PY.read_text()\nOP_SHA = hashlib.sha256(OP_PY.read_bytes()).hexdigest()\n_m = re.search(r"^(\\s*)SPLIT = (\\d+)\\s*$", SRC, re.M)\nassert _m, "SPLIT literal not found"\nINSTALLED = int(_m.group(2))\nORIG_SRC = __import__("inspect").getsource(M.OpMixin._split_cumalu)\nSPLIT_RE = re.compile(r"^(\\s*)SPLIT = (\\d+)\\s*$", re.M)\n\nlog("=" * 78); log("S0  PROVENANCE"); log("=" * 78)\nlog("  tinygrad      ", getattr(tinygrad, "__version__", "?"), "|", tinygrad.__file__)\nlog("  op.py sha256  ", OP_SHA)\nlog("  installed SPLIT", INSTALLED, "-> first two-stage N predicted at", 2*INSTALLED+1)\ntry:\n    log("  nvcc          ", subprocess.run(["nvcc","--version"],capture_output=True,text=True).stdout.strip().splitlines()[-1])\nexcept Exception as e:\n    log("  nvcc          unavailable", e)\n\n# ------------------------------------------------------------------ instrument\ndef require_fresh(t, what):\n    u = getattr(t, "uop", None)\n    if u is None: return\n    try: fresh = (not u.is_virtual) and (not u.has_buffer_identity())\n    except Exception: return\n    if not fresh:\n        raise RuntimeError(f"{what}: ALREADY REALIZED; realize() is a silent no-op "\n                           f"on a realized graph and timing it measures nothing.")\n\ndef dev_ms(tensor, iters=5):\n    from tinygrad.engine.realize import time_call\n    lin, vv = Tensor.linear_with_vars(tensor)\n    calls = list(lin.src)\n    if not calls:\n        raise RuntimeError("dev_ms: the LINEAR is EMPTY, so there is nothing to "\n                           "re-execute. Either the graph was already realized (a "\n                           "realize() on a realized graph is a silent no-op) or the "\n                           "expression compiled to nothing.")\n    for _ in range(2):\n        for c in calls: next(time_call(c, var_vals=vv))\n    tot = []\n    for _ in range(iters):\n        s = 0.0\n        for c in calls: s += next(time_call(c, var_vals=vv))\n        tot.append(s*1e3)\n    tot.sort()\n    med = statistics.median(tot)\n    if not med or med <= 0:\n        raise RuntimeError(f"dev_ms: median device time is {med} ms at {tensor.shape}. "\n                           f"A zero here is not a fast kernel, it is a measurement "\n                           f"that did not measure anything.")\n    return {"median_ms": med, "min_ms": tot[0], "max_ms": tot[-1], "n_kernels": len(calls)}\n\n# ------------------------------------------------------------------ the patch\nimport functools, textwrap\n_ORIG = Tensor._split_cumalu\n\ndef patch_split(k):\n    import tinygrad.codegen as CG\n    new_src, n = SPLIT_RE.subn(lambda m: f"{m.group(1)}SPLIT = {k}", ORIG_SRC, count=1)\n    if n != 1 or f"SPLIT = {k}" not in new_src:\n        raise RuntimeError(f"SPLIT substitution failed, n={n}")\n    ns = dict(vars(M))\n    exec(compile(textwrap.dedent(new_src), "<probe2:patched>", "exec"), ns)\n    Tensor._split_cumalu = functools.wraps(ns["_split_cumalu"])(\n        TWMOD._metadata_wrapper(ns["_split_cumalu"]))\n    CG.to_program_cache.clear()\n\ndef restore():\n    import tinygrad.codegen as CG\n    from tinygrad.engine import realize as RZ\n    Tensor._split_cumalu = _ORIG\n    CG.to_program_cache.clear()\n    for nm in ("runtime_cache","graph_cache","local_size_cache"):\n        if hasattr(RZ,nm): getattr(RZ,nm).clear()\n\n# ------------------------------------------------------------------ arms\nE = 1 << 24\ndef mk(r, n, seed=12):\n    return np.random.RandomState(seed).rand(r, n).astype(np.float32)\n\ndef relerr(a, want):\n    a = np.asarray(a, np.float64)\n    return float(np.abs(a-want).max()/max(1e-30, np.abs(want).max()))\n\ndef time_cumsum(x, want, iters=5):\n    t = Tensor(x).cumsum(-1)\n    require_fresh(t, "cumsum")\n    d = dev_ms(t, iters)\n    e = relerr(t.numpy(), want)\n    if e >= 1e-3: raise RuntimeError(f"GATE FAILED relerr={e:.3e}")\n    return d | {"relerr": e}\n\ndef time_copy(x, iters=5):\n    """The FLOOR, in this run, this instrument, this card.\n\n    A cumsum reads one float per element and writes one, and so does this. So it\n    is the honest denominator for "how close is this to the memory limit", and\n    `notes/33` makes the point and then quotes a floor measured in a different\n    session. Here it is the same loop, the same `time_call`, the same 30 seconds.\n\n    `Tensor(x).contiguous()` IS NOT A COPY. `x` is already contiguous, so\n    `contiguous()` returns a view of the input with no kernel behind it, the\n    LINEAR is empty, and the first version of this probe died with\n    `ZeroDivisionError: float division by zero` computing GB/s from a\n    `median_ms` of nothing. That is the fourth time this project has had a\n    measurement return a number derived from an absent computation; the fix is\n    to make the absent computation an explicit exception (see `dev_ms`) and to\n    use an expression that genuinely emits a kernel.\n\n    `x * 1.0` reads one float per element and writes one, through a real\n    elementwise kernel, and its correctness is checked against `x` so it cannot\n    be optimised into nothing without the gate noticing.\n    """\n    t = Tensor(x) * 1.0\n    require_fresh(t, "copy")\n    d = dev_ms(t, iters)\n    if relerr(t.numpy(), x) > 0:\n        raise RuntimeError("the floor arm did not reproduce its input")\n    nbytes = x.size*4\n    d["gbps"] = round(2*nbytes/d["median_ms"]/1e6, 1)\n    d["mib"] = round(nbytes/2**20, 1)\n    return d\n\n# ============================================================ A. the floor\nlog(""); log("=" * 78); log("A  THE COPY FLOOR, measured here, now"); log("=" * 78)\nxf = mk(E//4096, 4096)\nfl = time_copy(xf)\nlog(f"  copy E={xf.size} ({fl[\'mib\']:.0f} MiB in, same out)  {fl[\'median_ms\']:.3f} ms"\n    f"  {fl[\'gbps\']} GB/s   [MEASURED, the denominator everything below is compared to]")\nlog(f"  [{left():.0f}s left]")\n\n# ============================================ B. the interleaved SPLIT ladder\n# The ladder goes BELOW the installed value and far above it, because the\n# question is not "does it move" (run 1 answered that) but "is the window the\n# whole cost" (SPLIT=1 answers that) and "does the model hold at the top"\n# (SPLIT=4096 = one stage at every N in the ladder).\nLADDER = [1, 8, 64, 256, 1024, 4096]\nNS = [1024, 4096]\nROUNDS = 3\n\nlog(""); log("=" * 78)\nlog("B  THE SPLIT LADDER, INTERLEAVED")\nlog("   every arm measured in ABABAB order, %d rounds, one process, one card" % ROUNDS)\nlog("   so drift hits all arms equally instead of hitting whichever ran first")\nlog("=" * 78)\n# EVERY key that the loop below touches must exist first, or the run dies on a\n# KeyError at the first floor row and every stage is lost. The first version\n# initialised only ("floor", 4096) and then asked for ("floor", 1024), so the\n# probe died ~40 s in having measured the floor -- the one number it was going to\n# divide everything by -- and reported nothing.\n#\n# This is the same class of defect as the `realize()` no-op and the two-stage\n# mislabel: a bookkeeping key that was never checked against the loop that\n# consumes it. It is caught here by construction rather than by luck, because\n# the dict comprehension is over the same `LADDER` and `NS` the loop iterates.\nacc = {(k, n): [] for k in ["floor"] + LADDER for n in NS}\nassert set(acc) >= {("floor", n) for n in NS} | {(k, n) for k in LADDER for n in NS}, \\\n    "the accumulator is missing a key the sweep will ask for"\n\ndef one_point(k, n):\n    """Time one (SPLIT, N) cell, and NEVER let one cell\'s failure kill the run.\n\n    A single cell raising -- a compile error at one shape, an OOM, a gate\n    rejection -- must cost that cell and nothing else. The first version let it\n    propagate, so one bad cell lost the entire sweep including every cell before\n    it. The sweep is 36 cells and they are independent measurements; a failure in\n    one of them is information, not a reason to discard the other 35.\n    """\n    r = E // n\n    x = mk(r, n)\n    want = np.cumsum(x.astype(np.float64), -1)\n    try:\n        if k == "floor":\n            return time_copy(x)\n        if k == INSTALLED:\n            restore()\n        else:\n            patch_split(k)\n        return time_cumsum(x, want)\n    finally:\n        restore()\n\nfor rnd in range(ROUNDS):\n    log("")\n    log(f"  --- round {rnd+1}/{ROUNDS} ---")\n    for n in NS:\n        try:\n            d = one_point("floor", n)\n            acc[("floor", n)].append(d["median_ms"])\n            log(f"    floor   N={n:<5} {d[\'median_ms\']:8.3f} ms  {d[\'gbps\']} GB/s")\n        except Exception as e:\n            log(f"    floor   N={n:<5} FAILED {type(e).__name__}: {str(e)[:110]}")\n        for k in LADDER:\n            if left() < 90:\n                log(f"    SPLIT={k:<5} SKIPPED, {left():.0f}s left"); continue\n            try:\n                d = one_point(k, n)\n            except Exception as e:\n                log(f"    SPLIT={k:<5} N={n:<5} FAILED {type(e).__name__}: {str(e)[:110]}")\n                continue\n            acc[(k, n)].append(d["median_ms"])\n            log(f"    SPLIT={k:<5} N={n:<5} {d[\'median_ms\']:8.3f} ms  {d[\'n_kernels\']} kernels"\n                f"  relerr {d[\'relerr\']:.1e}  [{left():.0f}s left]")\n        log("")\n\n# ------------------------------------------------------------------ verdict\nlog("=" * 78); log("VERDICT"); log("=" * 78)\n\ndef spread(v):\n    return (max(v)-min(v))/min(v) if v and min(v) else float("inf")\n\nN = 4096\nif not acc[("floor", N)]:\n    log("  the copy floor did not survive to the verdict; no ratio is computable.")\n    raise SystemExit(0)\nfloor_med = statistics.median(acc[("floor", N)])\nlog(f"  copy floor at N={N}, E=2^24 : {floor_med:.3f} ms   [MEASURED this run]")\nlog("")\nlog(f"  {\'SPLIT\':>7} {\'rounds (ms)\':>34} {\'median\':>9} {\'spread\':>8} {\'x floor\':>8}")\ntab = []\nfor k in LADDER:\n    v = acc[(k, N)]\n    if not v: continue\n    med = statistics.median(v)\n    tab.append({"split": k, "rounds": v, "median_ms": med, "spread": spread(v),\n                "x_floor": med/floor_med})\n    log(f"  {k:>7} {str([round(t,2) for t in v]):>34} {med:>9.3f} "\n        f"{spread(v)*100:>7.1f}% {med/floor_med:>8.2f}")\n\nlog("")\nlog("  Q1  is the window the WHOLE cost?  SPLIT=1 is a window of 1 element, i.e.")\nlog("      no re-summation at all, which is what a scan does.")\nif tab:\n    t1 = next((r for r in tab if r["split"] == 1), None)\n    t256 = next((r for r in tab if r["split"] == INSTALLED), None)\n    if t1 and t256:\n        log(f"      SPLIT=1   {t1[\'median_ms\']:8.3f} ms  = {t1[\'x_floor\']:.2f}x the copy floor")\n        log(f"      SPLIT=256 {t256[\'median_ms\']:8.3f} ms  = {t256[\'x_floor\']:.2f}x the copy floor")\n        log(f"      -> shrinking the window from 256 to 1 buys only "\n            f"{t256[\'median_ms\']/t1[\'median_ms\']:.2f}x.")\n        log(f"      -> a windowless scan would still be {t1[\'x_floor\']:.1f}x the floor,"\n            f" so {\'SOME\' if t1[\'x_floor\']>1.5 else \'NOT MUCH\'} of the gap survives.")\nlog("")\nlog("  Q2  is the ladder monotone in SPLIT, with drift inside it?")\nif len(tab) >= 2:\n    meds = [r["median_ms"] for r in tab]\n    mono = all(meds[i] <= meds[i+1]*1.10 for i in range(len(meds)-1))\n    for r in tab:\n        log(f"      SPLIT={r[\'split\']:<6} {r[\'median_ms\']:8.3f} ms   round spread "\n            f"{r[\'spread\']*100:5.1f}%")\n    log(f"      monotone within 10%: {mono}")\n    worst_drift = max(r["spread"] for r in tab)\n    worst_effect = (tab[-1]["median_ms"]/tab[0]["median_ms"]) if tab[0]["median_ms"] else 0\n    log(f"      worst within-arm round spread : {worst_drift*100:.1f}%")\n    log(f"      full ladder effect (top/bottom): {worst_effect:.2f}x")\n    log(f"      effect / drift                : {worst_effect/(worst_drift or 1e-9):.1f}x")\n    log("      -> the effect is not a session difference; the interleave is what makes")\n    log("         that a statement rather than an assertion.")\n\nRESULT = {"probe": "axis/splitcausal_t4probe2.py", "dev": DEV,\n          "op_py_sha256": OP_SHA, "installed_SPLIT": INSTALLED,\n          "E": E, "NS": NS, "LADDER": LADDER, "ROUNDS": ROUNDS,\n          "floor_median_ms": floor_med,\n          "acc": {f"{k[0]}|{k[1]}": v for k, v in acc.items()},\n          "table_N4096": tab,\n          "wall_s": round(time.time()-T0, 1), "budget_s": BUDGET_S}\nopen(os.environ.get("SPLITCAUSAL2_OUT","splitcausal_t4b.json"),"w").write(\n    json.dumps(RESULT, indent=2, default=str))\nlog("")\nlog(f"wrote {os.environ.get(\'SPLITCAUSAL2_OUT\',\'splitcausal_t4b.json\')}"\n    f"   [{left():.0f}s of budget unused]")\n'
_SHA = 'fe0479c21bb3aed57e3707f3c075bfbc5bb11d26cec21cac9af4551fd5b9583e'
assert hashlib.sha256(_PAYLOAD.encode()).hexdigest() == _SHA
exec(compile(_PAYLOAD, '<splitcausal-t4-probe2>', 'exec'))
